# 🧩 G-LRAG Retrieval — Query Decomposition Edition (Google Colab)

Run the **G-LRAG hybrid retrieval** pipeline with the **pre-retrieval query
decomposition** mechanism ([`SubQueryRouter`](src/retrieval/sub_query_router.py))
on Google Colab.

> **What this notebook adds vs [`retrieval_colab_advanced.ipynb`](notebooks/retrieval_colab_advanced.ipynb):**
> Instead of the teammate's multi-query *variant* layer (keyword rewrite / HyDE /
> clause splits), this notebook uses the repo's production
> [`SubQueryRouter`](src/retrieval/sub_query_router.py) to **decompose a complex
> legal question into 2–4 independent sub-queries**, run the base
> [`HybridRetriever`](src/retrieval/retriever.py) once per sub-query, and fuse the
> per-sub-query rankings with RRF.

### Decomposition mechanism

| Step | Component | What it does |
|---|---|---|
| 1. Route | [`SubQueryRouter.route()`](src/retrieval/sub_query_router.py) | Decide whether the query should be split, using an injected LLM (optional) with a deterministic rule-based fallback. Returns a [`RouterDecision`](src/retrieval/sub_query_router.py) carrying `should_decompose` + `sub_queries`. |
| 2. Per-sub retrieval | [`HybridRetriever.retrieve()`](src/retrieval/retriever.py) | Run the full hybrid pipeline (lexical + dense + graph + rerank) once per sub-query. |
| 3. Fuse | RRF over `row_idx` | Reciprocal-rank-fuse the per-sub-query hit lists into one ranking. |
| 4. Output | `List[Hit]` | Trim to `FINAL_TOP_K`. Existing [`make_relevant_lists`](src/retrieval/retriever.py) and the F2 evaluation cell work unchanged. |

### Why this sits *on top of* `HybridRetriever`

The router is a **pure-Python, zero-heavy-dep** pre-processing step
([`sub_query_router.py`](src/retrieval/sub_query_router.py) imports only stdlib).
The downstream pipeline is the repo's production retriever, so each leg stays the
single source of truth. When the router decides **not** to decompose, this
notebook collapses to the exact behaviour of [`retrieval_colab.ipynb`](notebooks/retrieval_colab.ipynb).

> The LLM router path is **off by default** (`USE_LLM_ROUTER = False`), so the
> notebook runs with the rule-based fallback and needs no second GPU model. Flip
> `USE_LLM_ROUTER = True` to let a Qwen-style causal LM decide the split.


## 1. Setup — clone repo, mount Drive, install deps, configure

> ⚠️ **Edit the `Configuration` block** — especially `DATA_DIR`, and
> `USE_LLM_ROUTER` if you want an LLM to drive the decomposition decision.


In [ ]:
# ===== Configuration (edit these) =====================================
GITHUB_REPO = "https://github.com/vkb0205/Road2AI_ApplePie.git"
REPO_BRANCH = "main"
REPO_DIR    = "/workspace/Road2AI_ApplePie"

# Folder on your Google Drive that holds the Stage-6 artifacts + dev_set.
DATA_DIR = "/workspace/data"
DEV_DIR  = "/workspace/Road2AI_ApplePie/dev_set"

# --- Pipeline switches (legs) -----------------------------------------
USE_DENSE  = True    # load FAISS + BGE-m3 query encoder (runs on GPU + ~2.4GB)
USE_GRAPH  = True    # True -> load kg.gpickle for graph expansion
USE_RERANK = True    # cross-encoder rerank (BAAI/bge-reranker-v2-m3, on GPU)
FTS_MODE   = "bm25_ranked"   # 'bm25_ranked' (baseline) or 'fts_fast' (faster, weaker)

# --- NEW: Query decomposition mechanism -------------------------------
USE_QUERY_DECOMPOSITION = True   # pre-retrieval SubQueryRouter decomposition
USE_LLM_ROUTER          = True  # True -> use an LLM to decide decomposition
LLM_ROUTER_MODEL        = "Qwen/Qwen2.5-7B-Instruct"  # only if USE_LLM_ROUTER
MAX_SUB_QUERIES         = 4      # hard cap on sub-queries (2-4 recommended)
MIN_SUB_QUERY_CHARS     = 10     # discard sub-queries shorter than this
DEDUP_THRESHOLD         = 0.85   # token-overlap above which sub-queries merge

# --- Retrieval knobs --------------------------------------------------
RRF_K                 = 60     # RRF smoothing constant (base retriever)
BM25_TOPK             = 40     # lexical candidates
DENSE_TOPK            = 100    # dense candidates
CANDIDATE_TOPK        = 96     # fused candidates kept before rerank (per sub-query)
EXPANDED_TOP          = 96     # graph-expanded candidates (per sub-query)
SUB_QUERY_FINAL_TOP_K = 10     # hits produced per sub-query before cross-sub fusion
FUSE_K                = 60     # RRF smoothing across sub-query rankings
FINAL_TOP_K           = 10     # final hits returned after sub-query fusion

# --- Model download speedups (opt-in, big time saver) ------------------
FAST_DOWNLOAD     = True
HF_CACHE_ON_DRIVE = True
HF_TOKEN          = ""   # or set the Colab secret `HF_TOKEN` and leave this ''
HF_CACHE_DIR = "/workspace/data/cache_decomp"  # if HF_CACHE_ON_DRIVE

# --- Batch I/O (Section 8) --------------------------------------------
# Path to an OUTSIDE JSON file of queries to run in batch. Each record needs
# at least "id" and "question"; any extra fields are preserved in the output.
# Leave '' to fall back to dev_set/questions.json (runnable out of the box).
INPUT_QUERIES_PATH  = "/workspace/data/R2AIStage1DATA.json"                         # e.g. "/content/drive/.../my_queries.json"
# Path to the SEPARATE output file the notebook writes the results to.
OUTPUT_RESULTS_PATH = "/workspace/results_batch_decomposition.json"
# ======================================================================

# --- 0. GPU runtime check (fail fast) ---------------------------------
import subprocess, sys
gpu_ok = subprocess.run("nvidia-smi", shell=True,
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not gpu_ok:
    raise SystemExit(
        "❌ No GPU detected. This notebook is configured for a GPU runtime.\n"
        "   In Colab: Runtime → Change runtime type → T4 GPU, then restart and run all."
    )

!nvidia-smi -L
print("[gpu] GPU runtime confirmed.")

import os, sys, time, shutil
from pathlib import Path

# --- 1a. Clone the retrieval source code (lightweight, ~MB) -----------
if not Path(REPO_DIR).exists():
    print(f"[clone] {GITHUB_REPO} -> {REPO_DIR}")
    !git clone --depth 1 -b {REPO_BRANCH} {GITHUB_REPO} {REPO_DIR}
else:
    print(f"[clone] {REPO_DIR} already present")

SRC = Path(REPO_DIR) / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print("[src on path]", SRC)

# --- 1b. Big artifacts live on local disk under DATA_DIR (no Drive mount needed)

DATA = Path(DATA_DIR)
DEV  = Path(DEV_DIR)
print("[DATA_DIR exists]", DATA.exists(), DATA)
print("[DEV_DIR  exists]", DEV.exists(),  DEV)

# --- 1c. Install dependencies -----------------------------------------
!pip -q install pandas pyarrow networkx pyyaml python-dotenv psutil

if FAST_DOWNLOAD:
    !pip -q install hf_transfer
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if HF_CACHE_ON_DRIVE:
    os.makedirs(HF_CACHE_DIR, exist_ok=True)
    os.environ["HF_HOME"] = HF_CACHE_DIR
    os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE_DIR
    os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR

try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok: os.environ["HF_TOKEN"] = _tok
except Exception:
    if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN

print("[hf] transfer=" + ("on" if FAST_DOWNLOAD else "off")
      + " | cache=" + (HF_CACHE_DIR if HF_CACHE_ON_DRIVE else "ephemeral")
      + " | token=" + ("yes" if os.environ.get("HF_TOKEN") else "no"))

if USE_DENSE or USE_RERANK:
    print("[install] dense/rerank GPU deps (faiss-gpu, FlagEmbedding, torch+CUDA) ...")
    !pip -q install "faiss-gpu-cu12>=1.7.2" "FlagEmbedding>=1.2.10,<1.3" "numpy>=1.24" "transformers>=4.41,<4.46"
    !pip -q install torch --index-url https://download.pytorch.org/whl/cu121
    import torch as _t
    print("[torch]", _t.__version__, "| cuda available:", _t.cuda.is_available(),
          "| device:", _t.cuda.get_device_name(0) if _t.cuda.is_available() else "CPU")
    assert _t.cuda.is_available(), (
        "torch installed but CUDA not available. Make sure the Colab runtime "
        "is a GPU runtime (Runtime → Change runtime type → T4 GPU)."
    )

# LLM router needs a causal LM + tokenizer (only when USE_LLM_ROUTER).
if USE_LLM_ROUTER:
    print("[install] LLM router deps (accelerate, sentencepiece) ...")
    !pip -q install accelerate sentencepiece

print("[deps] ready")


## 2. Verify your input data is present

Run this to confirm the files you put on Drive are found before loading.


In [ ]:
required = {"chunk_store.sqlite": DATA}
if USE_DENSE:
    required.update({
        "faiss_index__BAAI_bge-m3.index": DATA,
        "chunk_meta_slim.parquet": DATA,
        "embed_model_meta__BAAI_bge-m3.json": DATA,
    })
if USE_GRAPH:
    required["kg.gpickle"] = DATA

ok = True
for name, d in required.items():
    p = d / name
    mb = (p.stat().st_size / 1e6) if p.exists() else 0
    flag = "OK " if p.exists() else "MISSING"
    if not p.exists(): ok = False
    print(f"{flag}  {p}  ({mb:.1f} MB)")

if DEV.exists():
    print(f"OK   {DEV/'questions.json'}")
    print(f"OK   {DEV/'ground_truth.json'}")
else:
    print(f"(optional dev_set not found at {DEV})")

assert ok, "❌ Missing required input files — upload them to DATA_DIR on Drive."


## 3. Build the base retriever (lexical + dense + graph legs)

This builds the repo's [`HybridRetriever`](src/retrieval/retriever.py) exactly as
[`retrieval_colab.ipynb`](notebooks/retrieval_colab.ipynb) does — the lexical FTS
leg, the (optional) dense FAISS leg (with the index moved onto the T4 GPU), the
(optional) graph expander, and the cross-encoder reranker (loaded lazily on first
use).

The decomposition layer (Section 4) wraps this object and reuses its `retrieve()`.


In [ ]:
from retrieval.bm25_index import FTSIndex
from retrieval.faiss_index import FAISSIndex, BGEQueryEncoder
from retrieval.graph_expand import GraphExpander
from retrieval.retriever import HybridRetriever, RetrievalConfig, make_relevant_lists

DB         = DATA / "chunk_store.sqlite"
FAISS_IDX  = DATA / "faiss_index__BAAI_bge-m3.index"
META       = DATA / "chunk_meta_slim.parquet"
MODEL_META = DATA / "embed_model_meta__BAAI_bge-m3.json"
KG         = DATA / "kg.gpickle"

# --- lexical leg (always on) ------------------------------------------
t0 = time.time()
fts = FTSIndex(str(DB), mode=FTS_MODE).open()
print(f"[fts] rows={fts.n_rows:,}  backend={fts.lexical_backend!r}  mode={FTS_MODE!r}  ({time.time()-t0:.1f}s)")

# --- GPU device used by the dense/rerank legs -------------------------
if USE_DENSE or USE_RERANK:
    import torch
    _dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[gpu] active device = {_dev}" + (f" ({torch.cuda.get_device_name(0)})" if _dev.type=='cuda' else ""))

# --- dense leg (optional, GPU) ----------------------------------------
faiss_index = None
query_encoder = None
if USE_DENSE:
    import psutil
    t0 = time.time()
    faiss_index = FAISSIndex(str(FAISS_IDX), str(META), str(MODEL_META)).load_index()
    _on_gpu = False
    if torch.cuda.is_available():
        try:
            import faiss as _faiss
            _res = _faiss.StandardGpuResources()
            _cpu_idx = faiss_index._index
            faiss_index._index = _faiss.index_cpu_to_gpu(_res, 0, _cpu_idx)
            del _cpu_idx; import gc; gc.collect()
            _on_gpu = True
        except Exception as _e:
            print(f"[dense] GPU move skipped ({_e}); keeping index on host")
    print(f"[dense] faiss ntotal={faiss_index.ntotal:,} dim={faiss_index.dim} on_gpu={_on_gpu} ({time.time()-t0:.1f}s)")
    if torch.cuda.is_available():
        print(f"[mem] GPU {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"[mem] host RSS {psutil.Process().memory_info().rss/1e9:.2f} GB")
    query_encoder = BGEQueryEncoder("BAAI/bge-m3", use_fp16=torch.cuda.is_available())
    print("[dense] BGE-m3 query encoder ready (downloads weights on first encode)")

# --- graph expander (optional) ----------------------------------------
graph_expander = None
if USE_GRAPH and KG.exists():
    t0 = time.time()
    graph_expander = GraphExpander.from_graph_and_meta(str(KG), str(META))
    print(f"[graph] expander loaded ({time.time()-t0:.1f}s)")
elif USE_GRAPH:
    print(f"[graph] kg.gpickle not found at {KG}; graph expansion disabled")

cfg = RetrievalConfig(
    use_dense=faiss_index is not None,
    use_reranker=USE_RERANK,
    top_bm25=BM25_TOPK,
    top_dense=DENSE_TOPK,
    rrf_k=RRF_K,
    fused_top=CANDIDATE_TOPK,
    expanded_top=EXPANDED_TOP,
    final_top_k=FINAL_TOP_K,
)
base_retriever = HybridRetriever(fts, faiss_index=faiss_index, graph_expander=graph_expander,
                                 query_encoder=query_encoder, config=cfg)
print("[retriever] base HybridRetriever ready")


## 4. Build the query decomposition router

Constructs the repo's [`SubQueryRouter`](src/retrieval/sub_query_router.py).

* **`USE_LLM_ROUTER = False`** (default): the router uses the deterministic
  rule-based fallback ([`rule_based_decompose`](src/retrieval/sub_query_router.py))
  — splits on semicolons and legal conjunctions (`và`, `cần xác định`, …), then
  normalises/dedups via [`normalize_sub_queries`](src/retrieval/sub_query_router.py).
  No extra model loaded.
* **`USE_LLM_ROUTER = True`**: a causal LM ([`LLM_ROUTER_MODEL`](#)) is loaded
  and wrapped as a `(prompt: str) -> str` callable injected into the router. The
  router sends [`DECOMPOSITION_PROMPT`](src/retrieval/sub_query_router.py) and
  parses the JSON `{should_decompose, sub_queries}`. On any failure it falls back
  to the rule-based path automatically.


In [ ]:
from retrieval.sub_query_router import (
    SubQueryRouter,
    SubQueryRouterConfig,
    RouterDecision,
    rule_based_decompose,
    normalize_sub_queries,
    DECOMPOSITION_PROMPT,
)
import retrieval.sub_query_router as _sqr_mod
_EXTRA_DECOMPOSITION_RULES = """\

Quy tắc tách sub-query (bổ sung):
- Mỗi sub-query phải là một truy vấn hoàn chỉnh, không phải mảnh câu.
- Mỗi sub-query phải giữ nguyên ngữ cảnh pháp lý và subject chính của ý đó.
- Nếu sub-query bị quá ngắn hoặc quá chung chung, hãy mở rộng bằng cách thêm đối tượng, hành vi, hoặc bối cảnh pháp lý liên quan.
- Không được tạo sub-query chỉ là cụm danh từ hay câu cụt.
- Ưu tiên dạng: [hành vi / vấn đề pháp lý] + [đối tượng / bối cảnh] + [mục tiêu truy vấn].

Ví dụ sub-query tốt (đầy đủ, độc lập, có legal anchor):
- Xác định hành vi xâm phạm quyền tác giả khi đối thủ sao chép trái phép phần mềm để cho thuê thu lợi.
- Cần chuẩn bị tài liệu và chứng cứ gì khi gửi đơn yêu cầu xử lý hành vi xâm phạm quyền tác giả đối với phần mềm.
- Cách tính thiệt hại và mất cơ hội kinh doanh do hành vi xâm phạm quyền tác giả đối với phần mềm."""
_sqr_mod.DECOMPOSITION_PROMPT = _sqr_mod.DECOMPOSITION_PROMPT.replace(
    "Quy tắc:", _EXTRA_DECOMPOSITION_RULES + "\n\nQuy tắc:", 1)
# Keep the notebook-local import name in sync too.
DECOMPOSITION_PROMPT = _sqr_mod.DECOMPOSITION_PROMPT
print("[router] DECOMPOSITION_PROMPT tightened (notebook-local override)")


def _build_llm_call(model_name):
    """Load a causal LM and return a (prompt: str) -> str callable for the router."""
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto' if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )
    model.eval()
    print(f"[llm_router] model ready on "
          f"{'cuda' if torch.cuda.is_available() else 'cpu'} ({model_name})")

    def llm_call(prompt: str) -> str:
        inputs = tok(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.2,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tok.eos_token_id,
                eos_token_id=tok.eos_token_id,
            )
        gen = out[0][inputs['input_ids'].shape[1]:]
        return tok.decode(gen, skip_special_tokens=True)

    return llm_call


llm_call = None
if USE_LLM_ROUTER:
    llm_call = _build_llm_call(LLM_ROUTER_MODEL)

router = SubQueryRouter(
    llm_call=llm_call,
    config=SubQueryRouterConfig(
        use_llm=USE_LLM_ROUTER,
        max_sub_queries=MAX_SUB_QUERIES,
        min_sub_query_chars=MIN_SUB_QUERY_CHARS,
        dedup_threshold=DEDUP_THRESHOLD,
        debug=True,
    ),
)
print("[router] SubQueryRouter ready "
      f"(use_llm={USE_LLM_ROUTER}, max_sub_queries={MAX_SUB_QUERIES}, "
      f"min_chars={MIN_SUB_QUERY_CHARS}, dedup={DEDUP_THRESHOLD})")

# Quick sanity check on the example query so you can see the decision before
# running the full retriever.
_demo_q = ("đối thủ sao chép trái phép phần mềm để cho thuê thu lợi và làm mất khách hàng; "
           "cần xác định hành vi xâm phạm quyền tác giả ở điểm nào, cách tính tổn thất về "
           "cơ hội kinh doanh ra sao, và phải chuẩn bị tài liệu / chứng cứ gì khi gửi đơn "
           "yêu cầu xử lý?")
_demo_decision = router.route(_demo_q)
print("\n[demo] route decision:")
print("  source          :", _demo_decision.source)
print("  should_decompose:", _demo_decision.should_decompose)
print("  num_sub_queries :", _demo_decision.num_sub_queries)
for i, sq in enumerate(_demo_decision.sub_queries):
    print(f"  sub-query[{i}]   : {sq}")
if _demo_decision.fallback_log is not None:
    print("  fallback_reason :", _demo_decision.fallback_log.reason)
    print("  fallback_rule   :", _demo_decision.fallback_log.rule_triggered)


## 5. Decomposing retrieval layer

[`DecomposingHybridRetriever`](#) orchestrates the decomposition mechanism on top
of the base [`HybridRetriever`](src/retrieval/retriever.py):

1. `router.route(query)` → [`RouterDecision`](src/retrieval/sub_query_router.py).
2. If `should_decompose` is False (or only one sub-query) → single
   `base.retrieve(query)` path (identical to [`retrieval_colab.ipynb`](notebooks/retrieval_colab.ipynb)).
3. Otherwise → run `base.retrieve(sub_query)` once per sub-query (temporarily
   widening `final_top_k` to `SUB_QUERY_FINAL_TOP_K` so each leg returns enough
   candidates), then RRF-fuse the per-sub-query rankings by `row_idx`.
4. Return `List[Hit][:FINAL_TOP_K]`.

When `explain=True`, each sub-query is run via `base.debug_retrieve` and a list of
[`SubQueryTrace`](src/retrieval/debug.py) is stored on `last_sub_query_traces`
(one block per sub-query with the route decision + nested per-stage trace).


In [ ]:
from retrieval.retriever import Hit, make_relevant_lists
from retrieval.debug import (
    SubQueryTrace,
    RouterDebugLog,
    format_sub_query_trace as _fmt_sq_trace,
)
from collections import defaultdict
import re as _re


# --- Pre-retrieval sub-query cleanup (lightweight, no split-logic change) --
# Applied right after router.route() and before base.retrieve(sub_query) to
# stabilise the text the retriever sees: trim/collapse whitespace, drop
# fragments & near-duplicates, and backfill a minimal legal anchor from the
# original query when a sub-query is too generic. This protects the pipeline
# from 'câu cụt' / noun-phrase sub-queries regardless of router source.
_LEGAL_ANCHOR_TERMS = (
    "quyền tác giả", "phần mềm", "chứng cứ", "thiệt hại",
    "hành vi", "xâm phạm", "bồi thường", "trách nhiệm",
    "hợp đồng", "sở hữu trí tuệ", "tổn thất",
)
_ANCHOR_RE = _re.compile("|".join(_re.escape(t) for t in _LEGAL_ANCHOR_TERMS),
                        _re.IGNORECASE)


def clean_subquery(q: str) -> str:
    """Trim + collapse whitespace + strip stray leading/trailing punctuation."""
    q = " ".join(q.strip().split())
    q = q.strip(" .;,:-–—")
    return q


def _ensure_legal_anchor(q: str, context: str) -> str:
    """If a sub-query lacks any core legal phrase, graft the first anchor term
    found in the original query context onto the sub-query so the retriever
    has a legal hook. No-op if the sub-query already carries an anchor or the
    context has none."""
    if not q or _ANCHOR_RE.search(q):
        return q
    m = _ANCHOR_RE.search(context or "")
    if not m:
        return q
    anchor = m.group(0).lower()
    q = q.rstrip(" .")
    return f"{q} ({anchor})"


def normalize_subqueries(subqueries, context="", min_chars=10):
    """Lightweight normalisation applied right before retrieval:
      - clean whitespace / stray punctuation
      - drop sub-queries shorter than min_chars (fragments / câu cụt)
      - backfill a minimal legal anchor from context when too generic
      - dedup near-identical sub-queries (case-insensitive)
    Returns a list of stable, self-contained sub-query strings."""
    out, seen = [], set()
    for q in subqueries:
        q = clean_subquery(q)
        if len(q) < min_chars:
            continue
        q = _ensure_legal_anchor(q, context)
        key = " ".join(q.lower().split())
        if key in seen:
            continue
        seen.add(key)
        out.append(q)
    return out


class DecomposingHybridRetriever:
    """Pre-retrieval query-decomposition layer over HybridRetriever.

    Pipeline per query:
      1. SubQueryRouter.route(query) -> RouterDecision
      2. if not should_decompose -> base.retrieve(query) (single path)
      3. else: for each sub-query -> base.retrieve(sub_query) -> per-sub hits
      4. RRF-fuse per-sub rankings by row_idx -> ordered hits
      5. return List[Hit][:FINAL_TOP_K]

    Debug: when explain=True, each sub-query is run via base.debug_retrieve and a
    list[SubQueryTrace] is stored on last_sub_query_traces.
    """

    def __init__(self, base, router, use_decomposition=True,
                 sub_query_final_top_k=10, fuse_k=60):
        self.base = base
        self.router = router
        self.use_decomposition = use_decomposition
        self.sub_query_final_top_k = sub_query_final_top_k
        self.fuse_k = fuse_k
        self.last_decision = None
        self.last_sub_query_traces = None

    @property
    def last_trace_formatted(self):
        if self.last_sub_query_traces:
            return _fmt_sq_trace(self.last_sub_query_traces)
        return None

    # ------------------------------------------------------------------ #
    def _route_decision_log(self, decision, query):
        fl = decision.fallback_log
        return RouterDebugLog(
            original_query=query,
            should_decompose=decision.should_decompose,
            num_sub_queries=decision.num_sub_queries,
            sub_queries=decision.sub_queries,
            source=decision.source,
            raw_llm_output=decision.raw_llm_output,
            fallback_reason=fl.reason if fl else "",
            fallback_rule=fl.rule_triggered if fl else "",
        )

    def _fuse_hits(self, per_sub_hits):
        """RRF-fuse per-sub-query Hit lists by row_idx.

        Score contribution of a hit at 1-based rank r in one sub-query is
        1 / (fuse_k + r). The representative Hit kept per row_idx is the one
        with the highest original (rerank) score; its score is overwritten with
        the fused RRF score.
        """
        scores = defaultdict(float)
        payload = {}
        for hits in per_sub_hits:
            for rank, h in enumerate(hits, start=1):
                ri = int(h.row_idx)
                scores[ri] += 1.0 / (self.fuse_k + rank)
                if ri not in payload or h.score > payload[ri].score:
                    payload[ri] = h
        ordered = sorted(scores.items(), key=lambda x: (-x[1], x[0]))
        out = []
        for ri, sc in ordered:
            h = payload[ri]
            h.score = float(sc)
            out.append(h)
        return out

    # ------------------------------------------------------------------ #
    def retrieve(self, query, fetch_text=False, explain=False, print_trace=False):
        cfg = self.base.config
        prev_final = cfg.final_top_k
        traces = []
        try:
            decision = self.router.route(query)
            self.last_decision = decision
            # Pre-retrieval cleanup: normalise each sub-query, drop fragments /
            # duplicates, backfill a minimal legal anchor from the original query
            # context. Does NOT change the router's split logic — only stabilises
            # the text the retriever actually sees.
            cleaned = normalize_subqueries(decision.sub_queries, context=query,
                                           min_chars=MIN_SUB_QUERY_CHARS)
            decision.sub_queries = cleaned
            decision.num_sub_queries = len(cleaned)
            print("[decomposition] source=%s should_decompose=%s num_sub_queries=%d" % (
                decision.source, decision.should_decompose, decision.num_sub_queries))
            for i, sq in enumerate(decision.sub_queries):
                print(f"  sub-query[{i}]: {sq}")

            no_split = ((not self.use_decomposition)
                        or (not decision.should_decompose)
                        or (len(decision.sub_queries) <= 1))

            if no_split:
                # Single path — identical to retrieval_colab.ipynb.
                if explain:
                    hits = self.base.debug_retrieve(query, fetch_text=fetch_text, print_trace=False)
                else:
                    hits = self.base.retrieve(query, fetch_text=fetch_text)
                docs, articles = make_relevant_lists(hits)
                traces.append(SubQueryTrace(
                    original_query=query,
                    sub_query_text=query,
                    sub_query_index=0,
                    num_sub_queries=1,
                    route_decision=self._route_decision_log(decision, query),
                    retrieval_trace=self.base.last_trace if explain else None,
                    final_hits=[h.to_dict() for h in hits],
                    relevant_docs=docs,
                    relevant_articles=articles,
                ))
                self.last_sub_query_traces = traces
                if print_trace:
                    print(_fmt_sq_trace(traces))
                return hits

            # Decompose path: widen final_top_k for per-sub-query runs.
            cfg.final_top_k = self.sub_query_final_top_k
            per_sub_hits = []
            for i, sq in enumerate(decision.sub_queries):
                if explain:
                    hits = self.base.debug_retrieve(sq, fetch_text=fetch_text, print_trace=False)
                else:
                    hits = self.base.retrieve(sq, fetch_text=fetch_text)
                per_sub_hits.append(hits)
                docs, articles = make_relevant_lists(hits)
                traces.append(SubQueryTrace(
                    original_query=query,
                    sub_query_text=sq,
                    sub_query_index=i,
                    num_sub_queries=decision.num_sub_queries,
                    route_decision=self._route_decision_log(decision, query),
                    retrieval_trace=self.base.last_trace if explain else None,
                    final_hits=[h.to_dict() for h in hits],
                    relevant_docs=docs,
                    relevant_articles=articles,
                ))
            self.last_sub_query_traces = traces

            fused = self._fuse_hits(per_sub_hits)
            out = fused[:prev_final]
            if print_trace:
                print(_fmt_sq_trace(traces))
            return out
        finally:
            cfg.final_top_k = prev_final


decomp_retriever = DecomposingHybridRetriever(
    base_retriever,
    router,
    use_decomposition=USE_QUERY_DECOMPOSITION,
    sub_query_final_top_k=SUB_QUERY_FINAL_TOP_K,
    fuse_k=FUSE_K,
)
print("[retriever] DecomposingHybridRetriever ready "
      f"(decomposition={USE_QUERY_DECOMPOSITION}, llm_router={USE_LLM_ROUTER}, "
      f"sub_query_top_k={SUB_QUERY_FINAL_TOP_K}, fuse_k={FUSE_K}, final_top_k={FINAL_TOP_K})")


## 6. Run a query

Edit `QUERY` and run. You'll see the router's decomposition decision printed
first, then the final top-K hits with metadata + chunk text, plus the collapsed
`relevant_docs` / `relevant_articles`.


In [ ]:
# ──────────────────────────  EDIT YOUR QUERY HERE  ──────────────────────────
QUERY = ("đối thủ sao chép trái phép phần mềm để cho thuê thu lợi và làm mất khách hàng; "
         "cần xác định hành vi xâm phạm quyền tác giả ở điểm nào, cách tính tổn thất về "
         "cơ hội kinh doanh ra sao, và phải chuẩn bị tài liệu / chứng cứ gì khi gửi đơn "
         "yêu cầu xử lý?")
# ────────────────────────────────────────────────────────────────────────────

TEXT_PREVIEW = 800   # chars of chunk_text to show per hit (0 = full text)

t0 = time.time()
hits = decomp_retriever.retrieve(QUERY, fetch_text=True)
print(f"\nRetrieved {len(hits)} hits in {time.time()-t0:.2f}s\n")

for i, h in enumerate(hits, 1):
    print(f"#{i}  row_idx={h.row_idx}  score={h.score:+.4f}  source={h.source}")
    print(f"    law_id    : {h.law_id}")
    print(f"    ten_van_ban: {h.ten_van_ban}")
    print(f"    dieu_so   : {h.dieu_so}")
    txt = h.chunk_text or ""
    if TEXT_PREVIEW and len(txt) > TEXT_PREVIEW:
        txt = txt[:TEXT_PREVIEW] + " …[truncated]"
    print(f"    chunk_text:\n{txt}")
    print("-" * 100)

docs, articles = make_relevant_lists(hits)
print("\nrelevant_docs:", docs)
print("relevant_articles:", articles)


## 6b. 🔬 Explainability — per-sub-query debug trace

`explain=True` records a [`SubQueryTrace`](src/retrieval/debug.py) per sub-query
(the route decision + the nested per-stage retrieval trace) and stores the list on
`decomp_retriever.last_sub_query_traces`. `print_trace=True` also prints it.

When the router decides **not** to decompose, you get a single sub-query block
whose nested trace is the normal single-query [`RetrievalTrace`](src/retrieval/debug.py).


In [ ]:
t0 = time.time()
expl_hits = decomp_retriever.retrieve(QUERY, fetch_text=True, explain=True, print_trace=True)
print(f"\nExplained retrieve: {len(expl_hits)} hits in {time.time()-t0:.2f}s")

traces = decomp_retriever.last_sub_query_traces
if traces:
    print("\nsub-query -> hits  docs  articles")
    for t in traces:
        print(f"  [{t.sub_query_index}/{t.num_sub_queries - 1}] "
              f"hits={len(t.final_hits)} docs={len(t.relevant_docs)} "
              f"articles={len(t.relevant_articles)}")
        print(f"     text={t.sub_query_text!r}")
        if t.relevant_articles:
            print("     relevant_articles:", t.relevant_articles)


## 6c. 🔬 Lexical vs Dense vs RRF — pre-expansion leg inspection

Read-only diagnosis of the traces captured in §6b. The question here is
whether broad / unwanted legal neighbours already enter the fused candidate
set at the **lexical + dense → RRF** stage — i.e. *before* graph expansion is
even considered.

For each sub-query it dumps the raw top items of the [`lexical`](src/retrieval/retriever.py)
leg (`bm25_score`), the [`dense`](src/retrieval/retriever.py) leg (`dense_score`) and the
[`rrf`](src/retrieval/rrf.py) fused set (`rrf_score`), then classifies every recorded
fused item by which leg(s) it came from:

- **lexical_only** — in the lexical recorded slice, not the dense one.
- **dense_only** — in the dense recorded slice, not the lexical one.
- **both_legs** — present in both legs (RRF boosts these most).
- **beyond_slice** — in neither recorded slice: it ranks deeper than the
  per-stage top-N the trace keeps, but still within a leg's full `count`
  (RRF can only draw from lexical + dense, so it must originate in one of
  them).

> Caveat: each trace stage only *records* the top `debug_top_n` items
> (default 8), while `count` holds the true full leg size. So `beyond_slice`
is not
> "graph noise" — it is a leg member that was too deep to be recorded.
> Graph expansion is **not** involved in this cell at all.

> This cell never mutates `decomp_retriever`, `base_retriever`, the router
> or any config — it only reads `last_sub_query_traces`.


In [ ]:
# ── 6c. Read-only inspection: lexical vs dense vs rrf legs ───────────────
# Uses the traces already captured by §6b. If they are missing (cell run
# out of order) we lazily re-run ONE explained retrieve so this cell is
# still runnable standalone. No retrieval/rerank/config code is modified.
from collections import OrderedDict

traces = getattr(decomp_retriever, "last_sub_query_traces", None)
if not traces:
    print("[inspect] no traces yet — running one explained retrieve …")
    decomp_retriever.retrieve(QUERY, fetch_text=True, explain=True, print_trace=False)
    traces = decomp_retriever.last_sub_query_traces

assert traces, "[inspect] no SubQueryTrace available — run §6b first."


def _stage(snapshot):
    """Return (rowidx_list_in_order, {row_idx: (score, meta_str)}) from a StageSnapshot."""
    if snapshot is None:
        return [], {}
    ids, score_map, meta_map = [], {}, {}
    for it in snapshot.top_items:
        ri = it.get("row_idx")
        if ri is None:
            continue
        ri = int(ri)
        ids.append(ri)
        sc = it.get("score")
        score_map[ri] = ("%.4f" % sc) if isinstance(sc, (int, float)) else "—"
        law = it.get("law_id", "") or ""
        ten = it.get("ten_van_ban", "") or ""
        dieu = it.get("dieu_so", "") or ""
        meta_map[ri] = f"[{law}|{ten}|{dieu}]"
    return ids, {ri: (score_map[ri], meta_map[ri]) for ri in ids}


def _dump(label, ids, info):
    print(f"  [{label}]")
    if not ids:
        print("     (no recorded items)")
        return
    for rank, ri in enumerate(ids):
        sc, meta = info[ri]
        print(f"     #{rank:<2} row_idx={ri:<6} score={sc:<8} {meta}")


print("=" * 92)
print("INSPECT: lexical  vs  dense  vs  rrf (fused)  — BEFORE graph expansion")
print("=" * 92)

# Aggregate fused-item leg-attribution across sub-queries.
global_attr = OrderedDict()  # row_idx -> list of (sub_query_index, bucket)

for t in traces:
    rt = t.retrieval_trace
    if rt is None:
        print(f"\n[sub {t.sub_query_index}] NO retrieval_trace — re-run §6b with explain=True")
        continue
    lex_snap = rt.stage("lexical")
    dense_snap = rt.stage("dense")
    rrf_snap = rt.stage("rrf")

    lex_ids, lex_info = _stage(lex_snap)
    dense_ids, dense_info = _stage(dense_snap)
    rrf_ids, rrf_info = _stage(rrf_snap)
    lex_set, dense_set, rrf_set = set(lex_ids), set(dense_ids), set(rrf_ids)

    print(f"\n— sub-query [{t.sub_query_index}/{t.num_sub_queries - 1}]  text={t.sub_query_text!r}")
    if lex_snap is not None:
        print(f"  lexical full count={lex_snap.count}  recorded={len(lex_ids)}  "
              f"skip={lex_snap.skip!r}")
    if dense_snap is not None:
        print(f"  dense   full count={dense_snap.count}  recorded={len(dense_ids)}  "
              f"skip={dense_snap.skip!r}")
    if rrf_snap is not None:
        diag = rrf_snap.diagnostics
        print(f"  rrf     full count={rrf_snap.count}  recorded={len(rrf_ids)}  "
              f"skip={rrf_snap.skip!r}  rankings_in={diag.get('rankings_in','?')} "
              f"fused_top={diag.get('fused_top','?')} k={diag.get('k','?')}")

    print("")
    _dump("lexical top items (bm25_score)", lex_ids, lex_info)
    print("")
    _dump("dense top items (dense_score)", dense_ids, dense_info)
    print("")
    _dump("rrf fused top items (rrf_score)", rrf_ids, rrf_info)

    # --- classify every recorded fused item by leg attribution -----------
    buckets = {"both_legs": [], "lexical_only": [], "dense_only": [], "beyond_slice": []}
    for ri in rrf_ids:
        in_lex = ri in lex_set
        in_dense = ri in dense_set
        if in_lex and in_dense:
            bucket = "both_legs"
        elif in_lex:
            bucket = "lexical_only"
        elif in_dense:
            bucket = "dense_only"
        else:
            bucket = "beyond_slice"
        buckets[bucket].append(ri)
        global_attr.setdefault(ri, []).append((t.sub_query_index, bucket))

    print("")
    print("  rrf item leg-attribution (recorded fused items):")
    for b, items in buckets.items():
        tag = "  <-- broad semantic neighbours likely enter here" if b == "dense_only" else ""
        print(f"     {b:<13} n={len(items):<3} {items}{tag}")

    # Which docs are in a leg but DROPPED by fusion (truncated to fused_top)?
    lex_dropped = lex_set - rrf_set
    dense_dropped = dense_set - rrf_set
    if lex_dropped:
        print(f"  lexical recorded items NOT in rrf (truncated by fused_top): {sorted(lex_dropped)}")
    if dense_dropped:
        print(f"  dense   recorded items NOT in rrf (truncated by fused_top): {sorted(dense_dropped)}")

    # --- last check: final relevant_docs / relevant_articles --------------
    print(f"  [final] hits={len(t.final_hits)}  relevant_docs={len(t.relevant_docs)}  "
          f"relevant_articles={len(t.relevant_articles)}")
    if t.relevant_articles:
        print(f"     relevant_articles: {t.relevant_articles}")
    if t.relevant_docs:
        print(f"     relevant_docs     : {t.relevant_docs}")

# --- Cross-sub-query summary: how do fused items get attributed? ----------
print("\n" + "=" * 92)
print("SUMMARY — rrf fused-item leg attribution aggregated across sub-queries")
print("=" * 92)
if not global_attr:
    print("No rrf items recorded.")
else:
    bucket_count = {"both_legs": 0, "lexical_only": 0, "dense_only": 0, "beyond_slice": 0}
    for ri, subs in global_attr.items():
        for _, b in subs:
            bucket_count[b] = bucket_count.get(b, 0) + 1
    total = sum(bucket_count.values())
    print(f"recorded rrf item-appearance across all sub-queries: {total}")
    for b, n in bucket_count.items():
        pct = (100.0 * n / total) if total else 0.0
        tag = "  <-- broad neighbours entering via the dense (semantic) leg" if b == "dense_only" else ""
        print(f"  {b:<13} n={n:<3} ({pct:4.1f}%){tag}")
    # Unique dense_only row_idx (the prime suspects for unwanted neighbours)
    dense_only_unique = sorted({ri for ri, subs in global_attr.items()
                               if any(b == "dense_only" for _, b in subs)})
    print(f"\nunique dense_only row_idx (suspect broad semantic neighbours): {dense_only_unique}")

print("\n[inspect] done — no retrieval/config code was modified.")


## 7. (Optional) Load a dev-set example question

If you uploaded `dev_set/questions.json`, pick one of the 20 questions by index.


In [ ]:
import json

if DEV.exists() and (DEV / "questions.json").exists():
    QUESTIONS = json.loads((DEV / "questions.json").read_text(encoding="utf-8"))
    for q in QUESTIONS:
        print(f"{q['id']:>2}. {q['question']}")
else:
    print(f"dev_set not found at {DEV}. Upload dev_set/questions.json to use the picker.")
    QUESTIONS = []

PICK = 1   # 1..20

if QUESTIONS:
    QUERY = QUESTIONS[PICK - 1]["question"]
    print("QUERY set to:\n", QUERY)
else:
    print("No dev questions loaded; set QUERY manually in the cell above.")


## 8. Batch run — outside JSON input → separate output file

Reads a list of queries from an **outside JSON file**
(``INPUT_QUERIES_PATH``, set in [§1](#)) — each record needs at least ``id``
and ``question`` (extra fields are preserved — falls back to
`dev_set/questions.json` when left blank). For every question it runs the
decomposition retrieval, assembles a grounded ``answer`` from the retrieved
passages (concatenated ``chunk_text`` + inline ``[N]`` citations), fills
``relevant_docs`` / ``relevant_articles``, and writes the full records to a
**separate output file** (``OUTPUT_RESULTS_PATH``).

The output record shape matches the grader contract:
`id` / `question` / `answer` / `relevant_docs` / `relevant_articles`.

> When a `ground_truth.json` is available, an F2 macro score is printed at the
> end. Compare it against the baseline / advanced notebooks to measure the
> gain from the pre-retrieval decomposition layer.


In [ ]:
# ======================================================================
# 8. Batch run — OUTSIDE JSON input  ->  SEPARATE output JSON file
# ======================================================================
# Reads queries from `INPUT_QUERIES_PATH` (an outside JSON file with the same
# shape as dev_set/questions.json: a list of {"id", "question", ...} records),
# runs the decomposition retriever for every question, assembles a grounded
# "answer" from the retrieved passages (concatenated chunk_text + inline [N]
# citations), fills relevant_docs / relevant_articles, and writes the full
# records to `OUTPUT_RESULTS_PATH`.
#
# Any extra fields already present on an input record (e.g. a pre-filled
# "answer") are preserved — only id + question are required. The retrieval-
# derived fields (answer, relevant_docs, relevant_articles) are always (re)built
# from the current retriever run.
import json as _json
from pathlib import Path

# --- resolve input / output paths -------------------------------------------
# Fall back to the dev_set questions.json when no explicit input file is given,
# so this cell is always runnable out of the box.
_INPUT_Q  = Path(INPUT_QUERIES_PATH) if str(INPUT_QUERIES_PATH).strip() else None
_OUTPUT_R = Path(OUTPUT_RESULTS_PATH)

if _INPUT_Q is None or not _INPUT_Q.exists():
    _DEV_LOCAL = Path(REPO_DIR) / "dev_set"
    _cand = (DEV / "questions.json") if (DEV / "questions.json").exists() \
            else (_DEV_LOCAL / "questions.json")
    _INPUT_Q = _cand
    print(f"[batch] INPUT_QUERIES_PATH not set / missing -> using {_INPUT_Q}")
else:
    print(f"[batch] input  : {_INPUT_Q}")
print(f"[batch] output : {_OUTPUT_R}")

assert _INPUT_Q.exists(), f"input query file not found: {_INPUT_Q}"
_OUTPUT_R.parent.mkdir(parents=True, exist_ok=True)


# --- answer builder: concatenate retrieved passages + inline citations -------
def _build_answer(hits, max_chars_per_hit=2000):
    """Retrieval-only answer (no generation LLM).

    Each hit's ``chunk_text`` is followed by a
    ``[N] {law_id} | {ten_van_ban} | {dieu_so}`` citation marker so every
    sentence can be traced back to its source. Matches the grader-expected
    record shape (id / question / answer / relevant_docs / relevant_articles).
    """
    parts = []
    for i, h in enumerate(hits, 1):
        text = (getattr(h, "chunk_text", "") or "").strip()
        if not text:
            continue
        if max_chars_per_hit and len(text) > max_chars_per_hit:
            text = text[:max_chars_per_hit] + " \u2026[truncated]"
        cite = f"[{i}] {getattr(h, 'law_id', '') or ''}"
        ten = str(getattr(h, "ten_van_ban", "") or "").strip()
        if ten:
            cite += f" | {ten}"
        dieu = str(getattr(h, "dieu_so", "") or "").strip()
        if dieu:
            cite += f" | {dieu}"
        parts.append(f"{text} {cite}")
    return " ".join(parts).strip()


# --- run the batch -----------------------------------------------------------
records_in = _json.loads(_INPUT_Q.read_text(encoding="utf-8"))
print(f"[batch] loaded {len(records_in)} queries")

records_out, t0 = [], time.time()
for rec in records_in:
    qid   = rec.get("id")
    query = rec.get("question") or rec.get("query") or ""
    if not query:
        print(f"  [skip] id={qid} has no question text")
        continue
    hits = decomp_retriever.retrieve(query, fetch_text=True)
    docs, articles = make_relevant_lists(hits)
    out = dict(rec)                       # preserve any extra fields from the input
    out["id"]                = qid
    out["question"]          = query
    out["answer"]            = _build_answer(hits)
    out["relevant_docs"]     = docs
    out["relevant_articles"] = articles
    records_out.append(out)
    print(f"  [done] id={qid}  hits={len(hits)}  docs={len(docs)}  articles={len(articles)}")

_OUTPUT_R.write_text(_json.dumps(records_out, ensure_ascii=False, indent=2),
                     encoding="utf-8")
print(f"\n[batch] wrote {len(records_out)} records to {_OUTPUT_R} "
      f"in {time.time()-t0:.1f}s")

# --- (optional) F2 score if a ground-truth file is available -----------------
try:
    _DEV_LOCAL = Path(REPO_DIR) / "dev_set"
    _GTPATH = ((DEV / "ground_truth.json") if (DEV / "ground_truth.json").exists()
               else (_DEV_LOCAL / "ground_truth.json"))
    if _GTPATH.exists():
        import sys as _sys
        if str(_DEV_LOCAL.parent) not in _sys.path:
            _sys.path.insert(0, str(_DEV_LOCAL.parent))
        from dev_set.eval import f2_macro
        _gt = _json.loads(_GTPATH.read_text(encoding="utf-8"))
        print(f"\nF2 macro (decomposition, batch) = "
              f"{f2_macro(records_out, _gt):.4f}  [gt={_GTPATH}]")
    else:
        print("\n(ground_truth.json not found - skipping F2 score)")
except Exception as _e:
    print(f"\n(F2 scoring skipped: {_e})")


## 9. Cleanup


In [ ]:
# ── Full memory teardown — release GPU + host resources before a fresh run ──
# Run this BEFORE re-running §1-§8 from scratch (or just do Kernel → Restart).
import gc, subprocess

def _safe_close(obj, attr, label):
    try:
        fn = getattr(obj, attr, None)
        if callable(fn):
            fn()
            print(f"[cleanup] {label} closed")
    except Exception as e:
        print(f"[cleanup] {label} close skipped ({e})")

# 1. Close the FTS SQLite connection (host RAM).
if 'fts' in globals():
    _safe_close(fts, 'close', 'fts')

# 2. Drop heavy GPU/host objects so their refcounts hit zero.
#    faiss_index._index (GPU), query_encoder (BGE-m3), base_retriever._reranker
#    (bge-reranker-v2-m3) and the LLM router model all live behind these names.
for _name in ('decomp_retriever', 'base_retriever', 'router',
              'query_encoder', 'faiss_index', 'graph_expander', 'fts'):
    if _name in globals():
        globals()[_name] = None
        print(f"[cleanup] {_name} released")
del _name

gc.collect()

# 3. Free CUDA: sync, empty the caching allocator, reset peak stats. Tensors /
#    the GPU FAISS index are reclaimed once the owning objects above are gone
#    and gc has run; empty_cache() returns freed blocks to the driver.
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        print(f"[cleanup] CUDA cache emptied | "
              f"allocated={torch.cuda.memory_allocated()/1e9:.2f} GB "
              f"reserved={torch.cuda.memory_reserved()/1e9:.2f} GB")
except Exception as e:
    print(f"[cleanup] torch CUDA cleanup skipped ({e})")

# 4. Verify the GPU is actually clear from the driver's perspective.
try:
    r = subprocess.run(
        "nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader,nounits",
        shell=True, capture_output=True, text=True)
    used, total = (float(x) for x in r.stdout.strip().split(","))
    print(f"[cleanup] nvidia-smi: {used:.0f} / {total:.0f} MiB used "
          f"({100*used/total:.1f}% of VRAM)")
except Exception:
    pass

print("\n✅ In-kernel memory cleared. For a guaranteed-fresh start "
      "(no stale Python state / cached models), also do: Kernel → Restart.")